In [ ]:
!pip install -q langgraph langchain langchain-core

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [ ]:
!pip install -q langgraph langchain-core tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class AttentionLayer(tf.keras.layers.Layer):

    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.b = self.add_weight(
            name="att_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True,
        )

    def call(self, x):
        scores = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

In [ ]:
import tensorflow as tf

class AttentionLayer(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.b = self.add_weight(
            name="att_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True,
        )

        super().build(input_shape)

    def call(self, x):

        scores = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        weights = tf.nn.softmax(scores, axis=1)

        return tf.reduce_sum(x * weights, axis=1)

    def get_config(self):
        config = super().get_config()
        return config

In [ ]:
from tensorflow.keras.models import load_model

MODEL_PATH = "/content/drive/MyDrive/ECG_Splits/DenseNet1D_clean.keras"

model = load_model(
    MODEL_PATH,
    custom_objects={
        "AttentionLayer": AttentionLayer
    }
)

In [ ]:
!ls "/content/drive/MyDrive/ECG_Splits"

accuracy_curve.png	   confusion_matrix.csv		  test.csv
best_densenet_10.keras	   confusion_matrix.png		  train.csv
classification_report.txt  DenseNet1D_clean.keras	  val.csv
clean_accuracy.txt	   DenseNet1D_labelnoise05.keras
clean_mitbih_dataset.csv   loss_curve.png


In [ ]:
import numpy as np

# ECG signal (360 values)
X_train = train_df.iloc[:, :360].values.astype(np.float32)
X_test  = test_df.iloc[:, :360].values.astype(np.float32)
X_val   = val_df.iloc[:, :360].values.astype(np.float32)

# Encoded labels
y_train = train_df["label_enc"].values.astype(np.int32)
y_test  = test_df["label_enc"].values.astype(np.int32)
y_val   = val_df["label_enc"].values.astype(np.int32)

# Reshape for DenseNet
X_train = X_train.reshape((-1, 360, 1))
X_test  = X_test.reshape((-1, 360, 1))
X_val   = X_val.reshape((-1, 360, 1))

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("Classes:", np.unique(y_train))

X_train: (69802, 360, 1)
y_train: (69802,)
Classes: [0 1 2 3]


In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9179


In [ ]:
class ECGState(TypedDict):

    # Input ECG
    ecg: np.ndarray

    # Prediction
    prediction: int
    probabilities: np.ndarray
    confidence: float
    entropy: float

    # Poison Detection
    attack_probability: float
    poison_flag: bool

In [ ]:
class PoisonDetectionAgent:
    def __init__(self, model, embedding_model, knn, maha_mean, maha_inv_cov,
                 confidence_threshold, entropy_threshold,
                 knn_threshold, mahalanobis_threshold):
        # store everything
        ...

    def __call__(self, state):
        ecg = np.expand_dims(state["ecg"], axis=0)
        probs = self.model.predict(ecg, verbose=0)[0]
        prediction = int(np.argmax(probs))
        confidence = float(np.max(probs))
        entropy = float(-np.sum(probs * np.log(probs + 1e-10)))

        emb = self.embedding_model.predict(ecg, verbose=0)
        knn_d = float(np.mean(self.knn.kneighbors(emb)[0]))
        maha_d = float(mahalanobis(emb.flatten(), self.maha_mean, self.maha_inv_cov))

        evidence = {
            "confidence":   confidence >= self.confidence_threshold,
            "entropy":      entropy    <= self.entropy_threshold,
            "knn":          knn_d      <= self.knn_threshold,
            "mahalanobis":  maha_d     <= self.mahalanobis_threshold,
            # add label_agreement here if you go route 2
        }
        clean = sum(evidence.values())
        attack_probability = (len(evidence) - clean) / len(evidence)

        return {
            "prediction": prediction, "probabilities": probs,
            "confidence": confidence, "entropy": entropy,
            "attack_probability": attack_probability,
            "poison_flag": attack_probability >= 0.5,
        }

In [ ]:
builder = StateGraph(ECGState)

builder.add_node(
    "PoisonDetection",
    PoisonDetectionAgent(model)
)

builder.add_edge(
    START,
    "PoisonDetection"
)

builder.add_edge(
    "PoisonDetection",
    END
)

graph = builder.compile()

In [ ]:
sample = X_test[25]

result = graph.invoke(
    {
        "ecg": sample
    }
)

print(result)

{'ecg': array([[ 1.73754498e-01],
       [ 1.54430643e-01],
       [ 1.43418685e-01],
       [ 1.39723361e-01],
       [ 1.40304103e-01],
       [ 1.43367425e-01],
       [ 1.49344847e-01],
       [ 1.58803791e-01],
       [ 1.70025736e-01],
       [ 1.78773195e-01],
       [ 1.79821998e-01],
       [ 1.68605000e-01],
       [ 1.42204508e-01],
       [ 9.99868438e-02],
       [ 4.46271040e-02],
       [-1.66007318e-02],
       [-7.19493777e-02],
       [-1.08262159e-01],
       [-1.16280422e-01],
       [-9.52553600e-02],
       [-5.36843538e-02],
       [-5.36346203e-03],
       [ 3.67857181e-02],
       [ 6.50452003e-02],
       [ 7.67776072e-02],
       [ 7.11979493e-02],
       [ 4.72525656e-02],
       [ 5.00437664e-03],
       [-5.14926240e-02],
       [-1.13290384e-01],
       [-1.68731630e-01],
       [-2.07533762e-01],
       [-2.24675596e-01],
       [-2.21752152e-01],
       [-2.04838336e-01],
       [-1.80704862e-01],
       [-1.54465139e-01],
       [-1.29722595e-01],
    

In [ ]:
for i, layer in enumerate(model.layers):
    print(i, layer.name, layer.output_shape if hasattr(layer, "output_shape") else "")

0 input_layer_2 
1 conv1d_30 
2 batch_normalization_32 
3 activation_32 
4 max_pooling1d_2 
5 batch_normalization_33 
6 activation_33 
7 conv1d_31 
8 concatenate_24 
9 batch_normalization_34 
10 activation_34 
11 conv1d_32 
12 concatenate_25 
13 batch_normalization_35 
14 activation_35 
15 conv1d_33 
16 concatenate_26 
17 batch_normalization_36 
18 activation_36 
19 conv1d_34 
20 concatenate_27 
21 batch_normalization_37 
22 activation_37 
23 conv1d_35 
24 average_pooling1d_4 
25 batch_normalization_38 
26 activation_38 
27 conv1d_36 
28 concatenate_28 
29 batch_normalization_39 
30 activation_39 
31 conv1d_37 
32 concatenate_29 
33 batch_normalization_40 
34 activation_40 
35 conv1d_38 
36 concatenate_30 
37 batch_normalization_41 
38 activation_41 
39 conv1d_39 
40 concatenate_31 
41 batch_normalization_42 
42 activation_42 
43 conv1d_40 
44 average_pooling1d_5 
45 batch_normalization_43 
46 activation_43 
47 conv1d_41 
48 concatenate_32 
49 batch_normalization_44 
50 activation_44 


In [ ]:
import tensorflow as tf

embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("dense_4").output
)

print("Embedding Shape:", embedding_model.output_shape)

Embedding Shape: (None, 128)


In [ ]:
sample = X_test[:1]

embedding = embedding_model.predict(sample, verbose=0)

print("Embedding Shape:", embedding.shape)
print(embedding)

Embedding Shape: (1, 128)
[[4.3389741e-02 0.0000000e+00 8.0535507e-01 0.0000000e+00 0.0000000e+00
  1.3640331e-01 0.0000000e+00 2.0026183e-02 1.3966325e-01 0.0000000e+00
  0.0000000e+00 0.0000000e+00 1.4089152e+00 1.2448301e+00 0.0000000e+00
  6.6238284e-02 8.6663818e-01 0.0000000e+00 9.9837023e-01 5.3802949e-01
  3.8269970e-01 1.3513095e+00 0.0000000e+00 3.2235840e-01 0.0000000e+00
  3.2872808e-01 8.9099848e-01 0.0000000e+00 0.0000000e+00 5.2195501e-01
  0.0000000e+00 0.0000000e+00 0.0000000e+00 1.6511828e-03 0.0000000e+00
  0.0000000e+00 1.6922247e+00 7.8753829e-02 0.0000000e+00 9.9544609e-01
  1.3920933e+00 0.0000000e+00 9.3953484e-01 1.0547789e+00 1.0593315e+00
  5.2088207e-01 1.3463345e-01 0.0000000e+00 6.8122637e-01 9.6892852e-01
  1.1268981e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  8.7686408e-01 7.0922476e-01 0.0000000e+00 8.5730851e-01 1.9641977e-02
  0.0000000e+00 5.8101803e-01 0.0000000e+00 7.6899457e-01 0.0000000e+00
  4.6365818e-01 1.1202779e+00 8.118476

In [ ]:
# EXTRACT EMBEDDINGS FOR ALL CLEAN ECG's
print("Extracting train embeddings...")

train_embeddings = embedding_model.predict(
    X_train,
    batch_size=256,
    verbose=1
)

print(train_embeddings.shape)

Extracting train embeddings...
273/273 ━━━━━━━━━━━━━━━━━━━━ 195s 704ms/step
(69802, 128)


In [ ]:
# Fit a KNN model
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=5,
    metric="euclidean"
)

knn.fit(train_embeddings)

print("KNN Ready")

KNN Ready


In [ ]:
# TEST SIMILARITY
test_embedding = embedding_model.predict(
    X_test[:1],
    verbose=0
)

distance, indices = knn.kneighbors(test_embedding)

print("Distances:", distance)
print("Nearest Neighbours:", indices)

Distances: [[1.02361512 1.02857828 1.03327751 1.05990171 1.13627386]]
Nearest Neighbours: [[37785 35950 36476 36539 37522]]


In [ ]:
class SimilarityAgent:

    def __init__(self, knn):

        self.knn = knn

    def analyze(self, embedding):

        distances, indices = self.knn.kneighbors(
            embedding
        )

        avg_distance = float(
            np.mean(distances)
        )

        return {

            "avg_knn_distance": avg_distance,

            "nearest_indices": indices[0].tolist(),

            "nearest_distances": distances[0].tolist()

        }

In [ ]:
similarity_agent = SimilarityAgent(knn)

result = similarity_agent.analyze(
    test_embedding
)

result

{'avg_knn_distance': 1.0563292980194092,
 'nearest_indices': [37785, 35950, 36476, 36539, 37522],
 'nearest_distances': [1.0236151218414307,
  1.028578281402588,
  1.0332775115966797,
  1.0599017143249512,
  1.1362738609313965]}

In [ ]:
class ConfidenceAgent:

    def analyze(self, probabilities):

        confidence = float(
            np.max(probabilities)
        )

        prediction = int(
            np.argmax(probabilities)
        )

        return {

            "prediction":prediction,

            "confidence":confidence

        }

In [ ]:
probabilities = model.predict(
    X_test[:1],
    verbose=0
)[0]

confidence_agent = ConfidenceAgent()

confidence_result = confidence_agent.analyze(
    probabilities
)

confidence_result

{'prediction': 2, 'confidence': 0.996323823928833}

In [ ]:
class EntropyAgent:

    def analyze(self, probabilities):

        entropy = float(

            -np.sum(

                probabilities *

                np.log(probabilities + 1e-10)

            )

        )

        return {

            "entropy":entropy

        }

In [ ]:
entropy_agent = EntropyAgent()

entropy_result = entropy_agent.analyze(
    probabilities
)

entropy_result

{'entropy': 0.02711990475654602}

In [ ]:
# Mahalanobis Distance Agent
import numpy as np
from scipy.spatial.distance import mahalanobis

In [ ]:
# Mean vector of all clean embeddings
embedding_mean = np.mean(train_embeddings, axis=0)

print("Mean Shape:", embedding_mean.shape)

Mean Shape: (128,)


In [ ]:
covariance_matrix = np.cov(train_embeddings, rowvar=False)

print(covariance_matrix.shape)

(128, 128)


In [ ]:
inverse_covariance = np.linalg.pinv(covariance_matrix)

print("Inverse Covariance Ready")

Inverse Covariance Ready


In [ ]:
class MahalanobisAgent:

    def __init__(self, mean, inv_cov):

        self.mean = mean
        self.inv_cov = inv_cov

    def analyze(self, embedding):

        distance = mahalanobis(
            embedding.flatten(),
            self.mean,
            self.inv_cov
        )

        return {
            "mahalanobis_distance": float(distance)
        }

In [ ]:
mahalanobis_agent = MahalanobisAgent(
    embedding_mean,
    inverse_covariance
)

result = mahalanobis_agent.analyze(
    test_embedding
)

result

{'mahalanobis_distance': 17.317364156960135}

In [ ]:
print("Computing Mahalanobis distances for training embeddings...")

train_mahalanobis = []

for embedding in train_embeddings:
    d = mahalanobis(
        embedding,
        embedding_mean,
        inverse_covariance
    )
    train_mahalanobis.append(d)

train_mahalanobis = np.array(train_mahalanobis)

print("Done!")

Computing Mahalanobis distances for training embeddings...
Done!


In [ ]:
print("Minimum :", np.min(train_mahalanobis))
print("Maximum :", np.max(train_mahalanobis))
print("Mean    :", np.mean(train_mahalanobis))
print("Std Dev :", np.std(train_mahalanobis))

Minimum : 4.099972183654678
Maximum : 61.35335398738656
Mean    : 10.259976787502648
Std Dev : 4.767708365067643


In [ ]:
mahalanobis_threshold = (
    np.mean(train_mahalanobis)
    + 3 * np.std(train_mahalanobis)
)

print("Threshold:", mahalanobis_threshold)

Threshold: 24.563101882705574


In [ ]:
class PoisonDetectionAgent:

    def __init__(self,
                 confidence_threshold=0.80,
                 entropy_threshold=0.50,
                 knn_threshold=1.50,
                 mahalanobis_threshold=24.56):

        self.confidence_threshold = confidence_threshold
        self.entropy_threshold = entropy_threshold
        self.knn_threshold = knn_threshold
        self.mahalanobis_threshold = mahalanobis_threshold

    def analyze(self,
                confidence,
                entropy,
                knn_distance,
                mahalanobis_distance):

        evidence = {}

        evidence["confidence"] = confidence >= self.confidence_threshold

        evidence["entropy"] = entropy <= self.entropy_threshold

        evidence["knn"] = knn_distance <= self.knn_threshold

        evidence["mahalanobis"] = (
            mahalanobis_distance <= self.mahalanobis_threshold
        )

        clean_votes = sum(evidence.values())
        poison_votes = 4 - clean_votes

        attack_probability = poison_votes / 4

        return {

            "attack_probability": attack_probability,

            "poison_flag": attack_probability >= 0.5,

            "clean_votes": clean_votes,

            "poison_votes": poison_votes,

            "evidence": evidence

        }

In [ ]:
poison_agent = PoisonDetectionAgent(
    mahalanobis_threshold=mahalanobis_threshold
)

result = poison_agent.analyze(
    confidence=0.9995,
    entropy=0.0046,
    knn_distance=1.056,
    mahalanobis_distance=17.317
)

result

{'attack_probability': np.float64(0.0),
 'poison_flag': np.False_,
 'clean_votes': np.int64(4),
 'poison_votes': np.int64(0),
 'evidence': {'confidence': True,
  'entropy': True,
  'knn': True,
  'mahalanobis': np.True_}}

In [ ]:
print("Computing KNN distances...")

train_knn_distances = []

for embedding in train_embeddings:

    distances, _ = knn.kneighbors(
        embedding.reshape(1, -1)
    )

    # Ignore the first neighbour (itself)
    avg_distance = np.mean(distances[0][1:])

    train_knn_distances.append(avg_distance)

train_knn_distances = np.array(train_knn_distances)

print("Done!")

Computing KNN distances...
Done!


In [ ]:
print("Mean :", np.mean(train_knn_distances))
print("Std  :", np.std(train_knn_distances))
print("Max  :", np.max(train_knn_distances))

In [ ]:
knn_threshold = (
    np.mean(train_knn_distances)
    + 3 * np.std(train_knn_distances)
)

print("KNN Threshold:", knn_threshold)

In [ ]:
print("Computing confidence distribution...")

train_probabilities = model.predict(
    X_train,
    batch_size=256,
    verbose=1
)

train_confidences = np.max(
    train_probabilities,
    axis=1
)

print("Done!")

In [ ]:
print("Mean :", np.mean(train_confidences))
print("Std  :", np.std(train_confidences))
print("Min  :", np.min(train_confidences))

In [ ]:
# cell 51
confidence_threshold = (
    np.mean(train_confidences)
    - 3 * np.std(train_confidences)
)

print(confidence_threshold)

In [ ]:
train_entropy = []

for probs in train_probabilities:

    entropy = -np.sum(
        probs * np.log(probs + 1e-10)
    )

    train_entropy.append(entropy)

train_entropy = np.array(train_entropy)

In [ ]:
print("Mean :", np.mean(train_entropy))
print("Std  :", np.std(train_entropy))

In [ ]:
entropy_threshold = (
    np.mean(train_entropy)
    + 3 * np.std(train_entropy)
)

print(entropy_threshold)